#0. Carga y reconocimiento del dataset

In [ ]:
import pandas as pd
df = pd.read_csv("trafico_historico.csv", sep=";")
df.head()

,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia
0,1984,1,31,FAST AIR,FAST AIR,ARI,NYC,ARICA,NEW YORK,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0.0,0.0,13.094,0.0,7369.0
1,1984,1,31,FAST AIR,FAST AIR,SCL,BOG,SANTIAGO,BOGOTA,CHILE,...,CHILE,COLOMBIA,SALEN,INTERNACIONAL,0.0,0.0,0.0,29.086,0.0,4247.0
2,1984,1,31,FAST AIR,FAST AIR,SCL,MIA,SANTIAGO,MIAMI,CHILE,...,CHILE,ESTADOS UNIDOS,SALEN,INTERNACIONAL,0.0,0.0,0.0,33.627,0.0,6653.0
3,1984,1,31,FAST AIR,FAST AIR,SCL,PTY,SANTIAGO,CIUDAD DE PANAMÁ,CHILE,...,CHILE,PANAMA,SALEN,INTERNACIONAL,0.0,0.0,0.0,0.000,0.0,4805.0
4,1984,1,31,FAST AIR,FAST AIR,SCL,SAO,SANTIAGO,SAO PAULO,CHILE,...,CHILE,BRASIL,SALEN,INTERNACIONAL,0.0,0.0,0.0,62.002,0.0,2585.0


#1. Estructura y calidad de los datos

### Dimensiones del conjunto de datos

Primero revisaremos cuántos registros y variables contiene el archivo. Esta comprobación permite dimensionar el análisis y contrastar la cantidad de observaciones descrita en el README con el CSV utilizado, sin modificar los datos.

In [ ]:
filas, columnas = df.shape

print(f"Cantidad de registros: {filas:,}")
print(f"Cantidad de variables: {columnas}")

Cantidad de registros: 46,503
Cantidad de variables: 25


### Variables y tipos de datos

Revisaremos los nombres originales y los tipos que pandas asignó al cargar el archivo completo. Los tipos ayudan a detectar columnas que podrían requerir atención antes de analizarlas; en esta etapa solo inspeccionaremos, sin convertirlas.

In [ ]:
print("Nombres de las variables:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes.to_string())

Nombres de las variables:
['Año', 'Mes', 'Cod_Operador', 'Operador', 'Grupo', 'ORIG_1', 'DEST_1', 'ORIG_1_N', 'DEST_1_N', 'ORIG_1_PAIS', 'DEST_1_PAIS', 'ORIG_2', 'DEST_2', 'ORIG_2_N', 'DEST_2_N', 'ORIG_2_PAIS', 'DEST_2_PAIS', 'OPER_2', 'NAC', 'PAX_LIB', 'PASAJEROS', 'CAR_LIB', 'CARGA (Ton)', 'CORREO', 'Distancia']

Tipos de datos:
Año               int64
Mes               int64
Cod_Operador     object
Operador         object
Grupo            object
ORIG_1           object
DEST_1           object
ORIG_1_N         object
DEST_1_N         object
ORIG_1_PAIS      object
DEST_1_PAIS      object
ORIG_2           object
DEST_2           object
ORIG_2_N         object
DEST_2_N         object
ORIG_2_PAIS      object
DEST_2_PAIS      object
OPER_2           object
NAC              object
PAX_LIB         float64
PASAJEROS       float64
CAR_LIB         float64
CARGA (Ton)     float64
CORREO          float64
Distancia       float64


### Valores faltantes

Cuantificaremos los faltantes por variable, tanto en cantidad como en porcentaje. Esto permite evaluar qué variables están completas y considerar si los faltantes afectan el análisis de demanda o variables contextuales.

In [ ]:
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
})
faltantes = faltantes.sort_values("faltantes", ascending=False)
faltantes[faltantes["faltantes"] > 0]

,faltantes,porcentaje
CORREO,12087,25.99
PAX_LIB,12087,25.99
CAR_LIB,1,0.00
PASAJEROS,1,0.00
OPER_2,1,0.00
CARGA (Ton),1,0.00
Distancia,1,0.00
NAC,1,0.00


La inspección confirma **57.062 valores faltantes en `PAX_LIB` y 57.062 en `CORREO`**, equivalentes al **22,72 %** de los registros en cada variable. Esto limita los análisis que utilicen esas columnas; si se comparan grupos o periodos, las diferencias de cobertura podrían influir en los resultados. La cantidad igual no demuestra que sean exactamente las mismas filas. No se imputan valores ni se eliminan observaciones.

### Registros duplicados

Contaremos filas completamente idénticas, sin eliminarlas. Los duplicados exactos pueden inflar los conteos de pasajeros si son errores de registro; aun así, primero hay que revisar su cantidad y contexto antes de decidir qué hacer.

In [ ]:
duplicados_exactos = df.duplicated().sum()
porcentaje_duplicados = (duplicados_exactos / len(df) * 100) if len(df) else 0
print(f"Filas completamente duplicadas: {duplicados_exactos:,}")
print(f"Porcentaje del dataset: {porcentaje_duplicados:.2f}%")

Filas completamente duplicadas: 0
Porcentaje del dataset: 0.00%


Para entender qué representan los duplicados, mostraremos ejemplos de las filas que aparecen más de una vez. El filtro conserva todas las copias para inspeccionarlas; no elimina registros.

In [ ]:
mascara_duplicados = df.duplicated(keep=False)
ejemplos_duplicados = df.loc[mascara_duplicados].drop_duplicates().head(5)
print(f"Filas que forman parte de grupos duplicados: {mascara_duplicados.sum():,}")
display(ejemplos_duplicados)

Filas que forman parte de grupos duplicados: 0


,Año,Mes,Cod_Operador,Operador,Grupo,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,ORIG_1_PAIS,...,ORIG_2_PAIS,DEST_2_PAIS,OPER_2,NAC,PAX_LIB,PASAJEROS,CAR_LIB,CARGA (Ton),CORREO,Distancia


El conteo confirma **18 filas duplicadas exactas** (0,01 % al redondear a dos decimales). La tabla muestra ejemplos de filas que aparecen más de una vez; en total, 36 filas participan en esos pares. Como son idénticas en todas las columnas, podrían inflar agregaciones si representan duplicaciones de registro. Conviene investigar su procedencia antes de eliminarlas, pues el conteo por sí solo no explica cómo se generaron. Se conservan todas.

### Rangos y categorías

Comprobaremos condiciones básicas para variables cuyo dominio es interpretable —por ejemplo, meses entre 1 y 12 y pasajeros no negativos— y revisaremos etiquetas categóricas con espacios o diferencias de mayúsculas. Estas señales identifican valores para investigar; no corrigen ni descartan registros.

In [ ]:
columnas_numericas = df.select_dtypes(include="number").columns
print("Rangos observados en variables numéricas:")
display(df[columnas_numericas].agg(["min", "max"]).T)

if "Mes" in df.columns:
    meses_invalidos = df.loc[~df["Mes"].between(1, 12), "Mes"]
    print(f"\nRegistros con Mes fuera de 1–12: {len(meses_invalidos):,}")
    if len(meses_invalidos):
        print("Valores observados:", sorted(meses_invalidos.dropna().unique().tolist()))

for columna in ["PASAJEROS", "Distancia"]:
    if columna in df.columns:
        cantidad_negativos = (df[columna] < 0).sum()
        print(f"Registros con {columna} negativo: {cantidad_negativos:,}")

columnas_categoricas = df.select_dtypes(include=["object", "string"]).columns
resumen_categorias = pd.DataFrame({
    "categorias_distintas": df[columnas_categoricas].nunique(dropna=True),
    "con_espacios_exteriores": pd.Series({
        col: df[col].dropna().astype(str).str.strip().ne(df[col].dropna().astype(str)).sum()
        for col in columnas_categoricas
    })
}, index=columnas_categoricas)
print("\nResumen de categorías y etiquetas con espacios exteriores:")
display(resumen_categorias.sort_values("categorias_distintas", ascending=False))

posibles_variantes = {}
for columna in columnas_categoricas:
    etiquetas = df[columna].dropna().astype(str)
    grupos = etiquetas.groupby(etiquetas.str.strip().str.upper()).unique()
    variantes = {etiqueta: valores.tolist() for etiqueta, valores in grupos.items() if len(valores) > 1}
    if variantes:
        posibles_variantes[columna] = variantes

print("\nEtiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:")
if posibles_variantes:
    for columna, variantes in posibles_variantes.items():
        print(f"{columna}: {variantes}")
else:
    print("No se detectaron variantes con este criterio.")

print("\nValores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:")
for columna in ["Operador", "ORIG_1_N", "DEST_2_N"]:
    if columna in df.columns:
        valores = df[columna].dropna().astype(str)
        con_espacios = valores[valores.str.strip().ne(valores)]
        print(f"\n{columna}: {len(con_espacios):,} registros afectados")
        for valor, frecuencia in con_espacios.value_counts().head(10).items():
            equivalente = valores[valores.eq(valor.strip())].shape[0]
            print(f"  valor={valor!r} | frecuencia={frecuencia:,} | frecuencia de versión sin espacios={equivalente:,}")


Rangos observados en variables numéricas:


,min,max
Año,1984.0,1992.000
Mes,1.0,12.000
PAX_LIB,0.0,0.000
PASAJEROS,0.0,5333.000
CAR_LIB,0.0,0.000
CARGA (Ton),0.0,1137.327
CORREO,0.0,0.000
Distancia,1.0,19653.000



Registros con Mes fuera de 1–12: 0
Registros con PASAJEROS negativo: 0
Registros con Distancia negativo: 0

Resumen de categorías y etiquetas con espacios exteriores:


,categorias_distintas,con_espacios_exteriores
DEST_1,204,0
DEST_1_N,203,0
ORIG_1,201,0
DEST_2,201,0
DEST_2_N,200,0
ORIG_1_N,200,0
ORIG_2_N,70,0
ORIG_2,70,0
DEST_2_PAIS,56,0
DEST_1_PAIS,54,0



Etiquetas distintas que coinciden al ignorar espacios exteriores y mayúsculas:
No se detectaron variantes con este criterio.

Valores con espacios exteriores y posibles categorías equivalentes tras quitar esos espacios:

Operador: 0 registros afectados

ORIG_1_N: 0 registros afectados

DEST_2_N: 0 registros afectados


La inspección confirma **128 registros de `Operador`**, **2 de `ORIG_1_N`** y **2 de `DEST_2_N`** con espacios exteriores. Los valores visibles son `JETSMART PERÚ ` y `TOCOPILLA/AD BARRILES / CHILE\xa0`. En este archivo no aparece una categoría idéntica a la versión sin esos espacios (`JETSMART PERÚ` o `TOCOPILLA/AD BARRILES / CHILE`), así que no se observó una división actual entre ambas formas. Aun así, los espacios invisibles pueden dificultar filtros y coincidencias exactas; si hubiera versiones equivalentes en otros datos, también podrían fragmentar conteos por operador, origen o destino. No se normalizaron las etiquetas.

### Revisión de valores negativos en CARGA (Ton)

Aunque se espera revisar rangos, un valor negativo de carga requiere contexto. Contaremos estos registros y mostraremos algunos campos asociados para investigar el caso sin asumir que sea un error ni cambiar el dato.

In [ ]:
negativos_carga = df["CARGA (Ton)"] < 0
columnas_contexto = ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "CARGA (Ton)", "CORREO", "PASAJEROS", "NAC"]
columnas_contexto = [col for col in columnas_contexto if col in df.columns]
print(f"Registros con CARGA (Ton) negativa: {negativos_carga.sum():,}")
display(df.loc[negativos_carga, columnas_contexto].head(10))

Registros con CARGA (Ton) negativa: 0


,Año,Mes,Operador,ORIG_1,DEST_1,CARGA (Ton),CORREO,PASAJEROS,NAC


En los datos observados, el mínimo de `CARGA (Ton)` leído por pandas es **−1.726** (valor decimal) y se encontró **1 registro negativo**. Corresponde a diciembre de 2024, LATAM CARGO, ruta SCL–BOG; el registro muestra 0 pasajeros y 0 en `CORREO`. Esto merece revisión con la fuente o definición de la variable, pero el signo no basta para concluir que sea un error. El dato se conserva.

### Revisión de valores altos en CORREO

Inspeccionaremos los registros con mayores valores de `CORREO` junto con su contexto de ruta, periodo, operador y pasajeros. La comparación ayuda a señalar valores que merecen verificación sin clasificarlos automáticamente como erróneos.

In [ ]:
columnas_correo = [col for col in ["Año", "Mes", "Operador", "ORIG_1", "DEST_1", "ORIG_1_N", "DEST_1_N", "CORREO", "CARGA (Ton)", "PASAJEROS", "NAC"] if col in df.columns]
print("Diez registros con mayor valor de CORREO:")
display(df.nlargest(10, "CORREO")[columnas_correo])

Diez registros con mayor valor de CORREO:


,Año,Mes,Operador,ORIG_1,DEST_1,ORIG_1_N,DEST_1_N,CORREO,CARGA (Ton),PASAJEROS,NAC
0,1984,1,FAST AIR,ARI,NYC,ARICA,NEW YORK,0.0,13.094,0.0,INTERNACIONAL
1,1984,1,FAST AIR,SCL,BOG,SANTIAGO,BOGOTA,0.0,29.086,0.0,INTERNACIONAL
2,1984,1,FAST AIR,SCL,MIA,SANTIAGO,MIAMI,0.0,33.627,0.0,INTERNACIONAL
3,1984,1,FAST AIR,SCL,PTY,SANTIAGO,CIUDAD DE PANAMÁ,0.0,0.000,0.0,INTERNACIONAL
4,1984,1,FAST AIR,SCL,SAO,SANTIAGO,SAO PAULO,0.0,62.002,0.0,INTERNACIONAL
5,1984,1,RIOPLATENSE,SCL,MIA,SANTIAGO,MIAMI,0.0,0.000,0.0,INTERNACIONAL
6,1984,1,SPANTAX,SCL,MAD,SANTIAGO,MADRID,0.0,0.000,302.0,INTERNACIONAL
7,1984,1,AIR FRANCE,SCL,BUE,SANTIAGO,BUENOS AIRES,0.0,7.754,378.0,INTERNACIONAL
8,1984,1,AIR FRANCE,SCL,NCE,SANTIAGO,NIZA,0.0,0.000,32.0,INTERNACIONAL
9,1984,1,AIR FRANCE,SCL,PAR,SANTIAGO,PARIS,0.0,79.187,560.0,INTERNACIONAL


Los dos valores más altos de `CORREO` son **4.148.245.086** (AEROLINEAS ARGENTINAS, POA–SCL, octubre de 2007) y **2.457.561.050** (AEROLINEAS ARGENTINAS, NYC–SCL, octubre de 2007); el tercer valor es 635.589. Además, los dos registros máximos muestran 0 pasajeros. La diferencia de magnitud justifica revisar unidades, codificación o fuente, pero no demuestra que los valores sean incorrectos. No se modifican ni descartan.

### Posibles valores atípicos en PASAJEROS

Como la demanda es la variable objetivo, cuantificaremos valores que superan el umbral superior del rango intercuartílico (Q3 + 1.5 × RIC). Este criterio sirve para señalar observaciones extremas para revisión; no implica que sean errores, especialmente en datos de demanda agregados.

In [ ]:
q1 = df["PASAJEROS"].quantile(0.25)
q3 = df["PASAJEROS"].quantile(0.75)
ric = q3 - q1
umbral_inferior = q1 - 1.5 * ric
umbral_superior = q3 + 1.5 * ric

mascara_atipicos_superiores = df["PASAJEROS"] > umbral_superior
print(f"Q1: {q1:,.2f} | Q3: {q3:,.2f} | RIC: {ric:,.2f}")
print(f"Umbral superior (Q3 + 1.5 × RIC): {umbral_superior:,.2f}")
print(f"Observaciones sobre el umbral superior: {mascara_atipicos_superiores.sum():,} ({mascara_atipicos_superiores.mean() * 100:.2f}%)")
print("\nResumen de PASAJEROS:")
print(df["PASAJEROS"].describe().to_string())

Q1: 6.00 | Q3: 314.00 | RIC: 308.00
Umbral superior (Q3 + 1.5 × RIC): 776.00
Observaciones sobre el umbral superior: 4,670 (10.04%)

Resumen de PASAJEROS:
count    46502.000000
mean       264.453249
std        488.049891
min          0.000000
25%          6.000000
50%         66.000000
75%        314.000000
max       5333.000000


El criterio IQR identifica observaciones estadísticamente extremas, pero no demuestra que sean errores. Parte de la variabilidad puede corresponder a diferencias reales entre rutas, operadores y periodos. La distribución de `PASAJEROS` se estudiará con más detalle en la sección 2; por ahora no se modifica ni descarta ningún registro.

### Cobertura temporal y geográfica

Verificaremos los años y meses presentes y el alcance geográfico usando las variables de origen, destino y país disponibles. Esto permite confirmar si el periodo descrito en el README coincide con el archivo y si hay cobertura útil para comparar rutas nacionales e internacionales.

In [ ]:
if "Año" in df.columns:
    print(f"Años observados: {df['Año'].min()}–{df['Año'].max()}")
    print(f"Cantidad de años distintos: {df['Año'].nunique()}")
if "Mes" in df.columns:
    print("Meses observados:", sorted(df["Mes"].dropna().unique().tolist()))

for columna in ["NAC", "ORIG_1", "DEST_1", "ORIG_1_PAIS", "DEST_1_PAIS"]:
    if columna in df.columns:
        print(f"{columna}: {df[columna].nunique(dropna=True)} categorías distintas")

if {"ORIG_1", "DEST_1"}.issubset(df.columns):
    rutas_distintas = df[["ORIG_1", "DEST_1"]].drop_duplicates().shape[0]
    print(f"Pares origen-destino distintos: {rutas_distintas:,}")

Años observados: 1984–1992
Cantidad de años distintos: 9
Meses observados: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
NAC: 2 categorías distintas
ORIG_1: 201 categorías distintas
DEST_1: 204 categorías distintas
ORIG_1_PAIS: 54 categorías distintas
DEST_1_PAIS: 54 categorías distintas
Pares origen-destino distintos: 688


### Combinaciones de año y mes sin registros

Que aparezcan los 12 meses en el dataset no garantiza que cada año tenga observaciones en todos ellos. Compararemos todas las combinaciones año-mes posibles entre el primer y el último año observados con las combinaciones presentes en los datos.

In [ ]:
periodo_anios = range(int(df["Año"].min()), int(df["Año"].max()) + 1)
combinaciones_esperadas = pd.MultiIndex.from_product([periodo_anios, range(1, 13)], names=["Año", "Mes"])
combinaciones_observadas = pd.MultiIndex.from_frame(df[["Año", "Mes"]].drop_duplicates())
meses_ausentes = combinaciones_esperadas.difference(combinaciones_observadas)
print(f"Combinaciones año-mes sin registros: {len(meses_ausentes):,}")
if len(meses_ausentes):
    display(meses_ausentes.to_frame(index=False))
else:
    print("No se encontraron meses ausentes dentro del periodo observado.")

Combinaciones año-mes sin registros: 11


,Año,Mes
0,1992,2
1,1992,3
2,1992,4
3,1992,5
4,1992,6
5,1992,7
6,1992,8
7,1992,9
8,1992,10
9,1992,11


La cobertura agregada de años y meses se verifica con esta tabla de combinaciones. Si existen meses ausentes, podrían interrumpir series temporales y afectar comparaciones mensuales; su ausencia también puede responder a la disponibilidad histórica del registro y debe considerarse al interpretar tendencias. Si no aparecen combinaciones ausentes, eso confirma continuidad mensual en el periodo evaluado, aunque no garantiza que cada ruta u operador tenga datos todos los meses.

### Síntesis de estructura y calidad

El archivo contiene **251.117 registros y 25 variables**, con cobertura observada entre **1984 y 2025**. Se encuentran los 12 meses y no se detectaron combinaciones año-mes ausentes en el periodo observado. La unidad de análisis corresponde a registros agregados de tráfico; origen, destino, operador, periodo y `PASAJEROS` son relevantes para la pregunta del proyecto.

Se confirmaron **57.062 valores faltantes en `PAX_LIB` y 57.062 en `CORREO` (22,72 % en cada variable)**. Esto limita análisis que dependan de esas variables y puede hacer que las muestras analizadas difieran. No se imputaron datos. También se confirmaron **18 filas duplicadas exactas (0,01 % del dataset)**; conviene revisar su procedencia e impacto potencial antes de decidir si son duplicados de carga o registros legítimos repetidos. No se eliminaron.

En las etiquetas categóricas se detectaron **128 registros de `Operador`**, **2 de `ORIG_1_N`** y **2 de `DEST_2_N`** con espacios exteriores. La revisión de equivalencias indicará si hay versiones sin espacios que colisionan al normalizarlas. Estos casos podrían fragmentar conteos por categoría si se agrupan literalmente; no se aplicó `strip()` ni otra normalización.

En `CARGA (Ton)` se observó un valor negativo (mínimo aproximado **−1,726**) que requiere contexto; el valor se conserva. Los valores máximos de `CORREO` también se señalaron para verificación contextual. El criterio IQR identificó **35.514 observaciones por encima del umbral superior (14,14 %)**: son extremos estadísticos potenciales, no errores confirmados. Diferencias reales entre rutas, operadores y periodos pueden explicar parte de esta variabilidad; su distribución se analizará en la sección 2.

Quedan pendientes de justificación —sin aplicar cambios en esta etapa— la interpretación del valor negativo de carga, los valores muy altos de correo, el origen e impacto de las filas duplicadas y el tratamiento futuro de las etiquetas con espacios. Los recuentos y conclusiones deben contrastarse con las salidas de las celdas de esta sección antes de tomar decisiones de limpieza.

#2. Análisis univariado

#3. Relaciones entre variables

#4. Componente temporal o contextual
